# Модуль 10. Разбор кейсов: Реальный redocking 1IEP

**Модуль 10. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** Открытые 1IEP PDBQT/SDF, box и официальный выход; upstream commit и hash — в docking/provenance.json.

Выполнить официальный пример Vina и сравнить геометрию с подготовленным лигандом.

**Workflow:** Проверка данных → box → Vina seed42/cpu1 → mapped RMSD → интерпретация.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab10_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%2010_docking.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [ ]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab10/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


## 1. Подготовленные данные и координатная система

Используем официальный tutorial 1IEP с сохранённым provenance. Это уже подготовленные receptor/ligand PDBQT и box; данный запуск не воспроизводит подготовку из сырого PDB. Проверяем hash и параметры области поиска до оптимизации. Центр и размер box имеют единицы Å.

**Перед запуском:** Проверьте источник подготовленных файлов и систему координат.

In [ ]:
from docking import first_pose, mapped_rmsd
import shutil, subprocess
docking_data=DATA/'docking'
provenance=json.loads((docking_data/'provenance.json').read_text())
print(json.dumps(provenance,indent=2))
print((docking_data/'1iep_receptor.box.txt').read_text())
reference=docking_data/'1iep_ligand.pdbqt'
official=docking_data/'1iep_ligand_vina_out.pdbqt'


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Подготовленные данные и координатная система — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Явная команда Vina

Запускаем внешний Vina с seed, CPU и exhaustiveness в argv; stdout/stderr и файл поз сохраняем. Если бинарника нет, полный режим BLOCKED: готовый файл tutorial можно анализировать, но нельзя объявить новым docking run. Score — оценка скоринговой функции при этой подготовке, не измеренный Kd.

**Перед запуском:** Разберите официальный выход; при установленном Vina выполните собственный запуск.

In [ ]:
binary=shutil.which('vina')
if binary is None:
    raise RuntimeError('BLOCKED: установите Linux Vina в PATH; готовый output не заменяет запуск')
pose=OUT/'redocked.pdbqt'
command=[binary,'--receptor',str(docking_data/'1iep_receptor.pdbqt'),
         '--ligand',str(reference),'--config',str(docking_data/'1iep_receptor.box.txt'),
         '--seed','42','--cpu','1','--exhaustiveness','8','--num_modes','9','--out',str(pose)]
print('argv:',command)
process=subprocess.run(command,capture_output=True,text=True,timeout=600)
(OUT/'vina-stdout.txt').write_text(process.stdout)
(OUT/'vina-stderr.txt').write_text(process.stderr)
process.check_returncode()
assert pose.is_file()


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Явная команда Vina — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Что именно измеряет RMSD

Сопоставляем тяжёлые атомы по serial в координатной системе рецептора и усредняем квадраты смещений. Лиганд отдельно не совмещаем: иначе ошибка размещения исчезнет. Такой RMSD не перебирает симметрии и использует подготовленный референс. Для независимой экспериментальной валидации нужны исходная структура, подготовка и проверка atom mapping.

**Перед запуском:** Сравните score и RMSD по serial тяжёлых атомов, без независимого совмещения лиганда.

In [ ]:
a,_=first_pose(reference)
b,score=first_pose(pose)
assert a.keys()==b.keys(), 'Нужно явное сопоставление атомов'
# WHY: do not align the ligand separately; preserve receptor coordinates.
rmsd=float(np.sqrt(np.mean([np.sum((a[k]-b[k])**2) for k in sorted(a)])))
assert math.isfinite(rmsd) and math.isfinite(score)
artifact={'mode':'executed-vina','score':score,'rmsd_angstrom':rmsd,
          'rmsd_definition':'heavy atom serials, receptor frame, no symmetry correction',
          'official_reference_rmsd_angstrom':mapped_rmsd(reference,official),
          'command':command,'binary_sha256':hashlib.sha256(Path(binary).read_bytes()).hexdigest()}
(OUT/'docking.json').write_text(json.dumps(artifact,indent=2))
print(artifact)


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Что именно измеряет RMSD — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

Полный Vina run обязателен: argv, новые позы и logs, конечные score/RMSD. Без бинарника BLOCKED; значения seed42 не являются универсальным эталоном.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [ ]:
record={'module':10,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Vina: basic docking](https://autodock-vina.readthedocs.io/en/latest/docking_basic.html).

Перейдите к [lab10_practice.ipynb](lab10_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.